### np.unique

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def reshape(flat, shape):
    if len(shape) == 0:
        return flat[0]
    if len(shape) == 1:
        return list(flat)
    chunk = _product(shape[1:])
    return [reshape(flat[i * chunk:(i + 1) * chunk], shape[1:]) for i in range(shape[0])]


def py_unique(a, return_index=False, return_inverse=False, return_counts=False):

    shape = get_shape(a)
    flat  = flatten(a)
    n     = len(flat)

    # argsort the flat list to process elements in sorted order
    sorted_indices = sorted(range(n), key=lambda i: flat[i])

    unique_vals    = []
    unique_idx     = []
    counts         = []
    inverse        = [0] * n

    for pos in sorted_indices:
        val = flat[pos]
        if len(unique_vals) == 0 or val != unique_vals[-1]:
            unique_vals.append(val)
            unique_idx.append(pos)
            counts.append(1)
        else:
            counts[-1] += 1
        inverse[pos] = len(unique_vals) - 1

    result = [unique_vals]

    if return_index:
        result.append(unique_idx)
    if return_inverse:
        result.append(inverse)
    if return_counts:
        result.append(counts)

    if len(result) == 1:
        return result[0]
    return tuple(result)

### test cases 

In [2]:
print(py_unique([3, 1, 2, 1, 3]))
print(py_unique([1, 2, 3]))
print(py_unique([5, 5, 5]))
print(py_unique(['b', 'a', 'c', 'a']))
print(py_unique([3, 1, 2, 1, 3], return_index=True))
print(py_unique([3, 1, 2, 1, 3], return_inverse=True))
print(py_unique([3, 1, 2, 1, 3], return_counts=True))
print(py_unique([3, 1, 2, 1, 3], return_index=True, return_inverse=True, return_counts=True))
print(py_unique([[3, 1], [2, 1]]))
print(py_unique([[3, 1], [2, 1]], return_counts=True))
print(py_unique([]))
print(py_unique([42]))
print(py_unique([1, 2, 3], return_inverse=True))
print(py_unique([3, 1, 2, 1, 3], return_index=True, return_counts=True))

[1, 2, 3]
[1, 2, 3]
[5]
['a', 'b', 'c']
([1, 2, 3], [1, 2, 0])
([1, 2, 3], [2, 0, 1, 0, 2])
([1, 2, 3], [2, 1, 2])
([1, 2, 3], [1, 2, 0], [2, 0, 1, 0, 2], [2, 1, 2])
[1, 2, 3]
([1, 2, 3], [2, 1, 1])
[]
[42]
([1, 2, 3], [0, 1, 2])
([1, 2, 3], [1, 2, 0], [2, 1, 2])


In [3]:
print(py_unique([[1, 2], [3]]))

ValueError: Jagged array: first row shape (2,), row 1 shape (1,)

In [4]:

print(py_unique([1, "a"]))

TypeError: '<' not supported between instances of 'str' and 'int'

In [5]:
print(py_unique(py_unique))

[<function py_unique at 0x000001F88803ECA0>]
